In [0]:
from pyspark.sql import functions as F
import numpy as np, pandas as pd, os

SEED = 42
N_BATCHES = 10
BATCH_DATES = [str(d.date()) for d in pd.date_range("2026-01-01", periods=N_BATCHES)]
OUT = "/Volumes/fintxn/ops/sim_batches"
TXN_COLS = ["trans_num", "trans_date_trans_time", "customer_id", "merchant",
            "category", "amt", "merch_lat", "merch_long", "is_fraud"]

spark.sql("CREATE VOLUME IF NOT EXISTS fintxn.ops.sim_batches")
print(BATCH_DATES)

In [0]:
src = spark.table("fintxn.ops.src_transactions")
epoch = F.unix_timestamp(F.to_timestamp("trans_date_trans_time", "yyyy-MM-dd HH:mm:ss"))

b = src.select(F.min(epoch).alias("lo"), F.max(epoch).alias("hi")).first()
lo, span = b.lo, b.hi - b.lo

plan = (src
    .withColumn("time_chunk", F.least(F.lit(N_BATCHES - 1),
        F.floor((epoch - F.lit(lo)) / F.lit(span) * N_BATCHES)).cast("int"))
    .withColumn("r",  F.rand(SEED))
    .withColumn("r2", F.rand(SEED + 1))
    .withColumn("defect_type", F.expr("""
        CASE WHEN r < 0.0100 THEN 'duplicate'
             WHEN r < 0.0150 THEN 'correction'
             WHEN r < 0.0175 THEN 'null_amount'
             WHEN r < 0.0200 THEN 'negative_amount'
             WHEN r < 0.0250 THEN 'bad_timestamp_format'
             WHEN r < 0.0270 THEN 'unknown_customer'
             WHEN r >= 0.0300 AND r < 0.0500 THEN 'late_arrival'
        END"""))
    .withColumn("late_offset", F.when(F.col("defect_type") == "late_arrival",
        1 + F.floor(F.col("r2") * 3)).otherwise(0).cast("int"))
    .withColumn("delivery_batch", F.when(
        F.col("time_chunk") + F.col("late_offset") <= N_BATCHES - 1,
        F.col("time_chunk") + F.col("late_offset")).otherwise(F.col("time_chunk")))
    # Cancel defects that can't happen: late rows that couldn't be pushed later,
    # corrections whose follow-up batch doesn't exist
    .withColumn("defect_type", F.expr("""
        CASE WHEN defect_type = 'late_arrival' AND delivery_batch = time_chunk THEN NULL
             WHEN defect_type = 'correction'  AND delivery_batch = 9 THEN NULL
             ELSE defect_type END"""))
    .drop("r", "r2", "late_offset"))

plan.write.mode("overwrite").saveAsTable("fintxn.ops.sim_plan")
display(spark.table("fintxn.ops.sim_plan").groupBy("delivery_batch").count().orderBy("delivery_batch"))

In [0]:
plan = spark.table("fintxn.ops.sim_plan")
d = F.col("defect_type")

def write_single_csv(df, folder, filename):
    tmp = f"{OUT}/_tmp"
    df.coalesce(1).write.mode("overwrite").option("header", "true").csv(tmp)
    part = [f.path for f in dbutils.fs.ls(tmp) if f.name.startswith("part-")][0]
    dbutils.fs.mkdirs(f"{OUT}/{folder}")
    dbutils.fs.mv(part, f"{OUT}/{folder}/{filename}")
    dbutils.fs.rm(tmp, recurse=True)

for i in range(N_BATCHES):
    day = plan.where(F.col("delivery_batch") == i)

    body = (day
        .withColumn("amt", F.when(d == "null_amount", F.lit(None).cast("string"))
                            .when(d == "negative_amount", F.concat(F.lit("-"), F.col("amt")))
                            .otherwise(F.col("amt")))
        .withColumn("trans_date_trans_time", F.when(d == "bad_timestamp_format",
            F.date_format(F.to_timestamp("trans_date_trans_time", "yyyy-MM-dd HH:mm:ss"),
                          "dd-MM-yyyy HH:mm"))
            .otherwise(F.col("trans_date_trans_time")))
        .withColumn("customer_id", F.when(d == "unknown_customer", F.lit("C99999"))
                                    .otherwise(F.col("customer_id"))))

    dups = body.where(d == "duplicate")
    corrections = (plan.where((d == "correction") & (F.col("delivery_batch") == i - 1))
        .withColumn("amt", F.round(F.col("amt").cast("double") * 1.1, 2).cast("string")))

    out = (body.select(TXN_COLS)
           .unionByName(dups.select(TXN_COLS))
           .unionByName(corrections.select(TXN_COLS)))

    if i == 6:   # batch 7: upstream adds a new column without warning
        out = out.withColumn("channel",
            F.when(F.rand(SEED) < 0.5, "online").otherwise("pos"))

    write_single_csv(out, "transactions", f"transactions_{BATCH_DATES[i]}.csv")
    print("wrote batch", i + 1, BATCH_DATES[i])

In [0]:
(spark.table("fintxn.ops.sim_plan")
    .where("defect_type IS NOT NULL")
    .select("trans_num", "defect_type", "time_chunk", "delivery_batch")
    .write.mode("overwrite").saveAsTable("fintxn.ops.injected_defects"))

display(spark.sql("""
    SELECT defect_type, count(*) AS rows
    FROM fintxn.ops.injected_defects
    GROUP BY defect_type ORDER BY defect_type"""))

In [0]:
cust = spark.table("fintxn.ops.src_customers").toPandas()
first = (spark.table("fintxn.ops.sim_plan")
         .groupBy("customer_id").agg(F.min("time_chunk").alias("first_chunk"))
         .toPandas())
cust = cust.merge(first, on="customer_id", how="left")
cust["first_chunk"] = cust["first_chunk"].fillna(0).astype(int)

rng = np.random.default_rng(SEED)
ADDR = ["street", "city", "state", "zip", "lat", "long", "city_pop"]
jobs = cust["job"].unique()
state = cust.copy()
changes = []
os.makedirs(f"{OUT}/customers", exist_ok=True)

for i in range(N_BATCHES):
    if i > 0:
        active = state.index[state["first_chunk"] <= i]
        for idx in rng.choice(active, size=max(1, int(len(active) * 0.02)), replace=False):
            cid = state.at[idx, "customer_id"]
            if rng.random() < 0.5:
                old = state.at[idx, "job"]
                state.at[idx, "job"] = rng.choice(jobs[jobs != old])
                changes.append((cid, BATCH_DATES[i], "job_change"))
            else:
                donor = rng.choice(state.index[state.index != idx])
                state.loc[idx, ADDR] = cust.loc[donor, ADDR].values
                changes.append((cid, BATCH_DATES[i], "address_change"))

    snap = state[state["first_chunk"] <= i].drop(columns=["first_chunk"])
    snap.to_csv(f"{OUT}/customers/customers_{BATCH_DATES[i]}.csv", index=False)

(spark.createDataFrame(pd.DataFrame(changes, columns=["customer_id", "snapshot_date", "change_type"]))
    .write.mode("overwrite").saveAsTable("fintxn.ops.injected_customer_changes"))
print(len(changes), "customer changes logged")

In [0]:
for folder in ["transactions", "customers"]:
    for f in dbutils.fs.ls(f"{OUT}/{folder}"):
        n = spark.read.option("header", "true").csv(f.path).count()
        print(f"{folder:13s} {f.name:35s} {n:>8,} rows")